# 10.5 · 数据增强 / Data Augmentation

> **课程定位 / Where this fits**
> 第 5 课，**Part 10 · 计算机视觉**。
> Lesson 5, **Part 10 · Computer Vision**.
>
> 再好的架构(CNN/ResNet)也需要**足够多样的数据**才能学好、不过拟合，但标注数据很贵。**数据增强(data augmentation)** 是个"免费午餐"：对训练图像做随机变换（翻转、裁剪、旋转、调色、Mixup、Cutout…），凭空造出大量"新"样本，**显著提升泛化、对抗过拟合**。它是 CV 实战的标配。本课**大量可视化**各种增强，并**实测它带来的提升**。
> Even great architectures (CNN/ResNet) need **diverse enough data** to learn well and not overfit, but labeled data is expensive. **Data augmentation** is a free lunch: apply random transforms (flip, crop, rotate, color, Mixup, Cutout…) to training images, conjuring many "new" samples that **boost generalization and fight overfitting**. Standard in CV practice. This lesson **heavily visualizes** augmentations and **measures the gain**.
>
> 💼 **实战/面试视角**："数据增强为什么有效 / 常用哪些 / Mixup/Cutout / 只增强训练集" 高频且实用。
> 💼 **Practical/interview angle:** "why augmentation works / common ones / Mixup/Cutout / augment only training set" — frequent and practical.

> 📐 **符号约定 / Notation**
> - 增强 —— 对输入图像的随机但保持标签的变换 / random, label-preserving transforms
> - $\lambda$ —— Mixup 的混合系数 / Mixup mixing coefficient

> 💡 **面试相关 / Interview-relevant**
> - "数据增强为什么能减轻过拟合"（出镜率 ★★★★★）
> - "为什么只对训练集增强、不对验证/测试集"（★★★★）
> - "Mixup / Cutout / CutMix 原理"（★★★★）
> - "增强相当于一种正则化"（★★★）

> 📦 **本课数据 / Data here**：增强**可视化**用 `skimage` 自带高清彩色样例图（无需下载、看得清楚）；**有无增强对比实验**用已缓存的 **FashionMNIST**（灰度，故用几何增强）。
> Visualizations use `skimage` built-in HD color samples (no download, clear); the with/without experiment uses cached **FashionMNIST** (grayscale → geometric augmentations).

---

## 学习目标 / Learning Objectives
1. 理解数据增强为什么有效（更多数据 + 注入不变性 = 正则化）。
   Understand why augmentation works (more data + injected invariances = regularization).
2. 掌握常用几何/颜色增强并**可视化**。
   Master and visualize common geometric/color augmentations.
3. **从零实现 Mixup 与 Cutout**。
   Implement Mixup and Cutout from scratch.
4. **实测**有无增强对过拟合与精度的影响。
   Empirically measure augmentation's effect on overfitting and accuracy.

## 目录 / TOC
1. [为什么增强有效 ⭐](#1)
2. [基础增强：翻转/裁剪/旋转/调色（可视化）⭐](#2)
3. [进阶：Mixup 与 Cutout（从零）⭐](#3)
4. [实测：有无增强对比 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 为什么增强有效 ⭐ / Why Augmentation Works

三个层面理解（面试可分点答）：
Three angles (good for structured interview answers):
1. **等于免费扩充数据**：模型见过的样本越多越不容易死记硬背。一张图水平翻转后还是同一类，却是一个"新"训练样本。
   **Free data expansion:** more samples → less memorization. A flipped image is the same class but a "new" example.
2. **注入不变性(invariance)**：我们**知道**"猫左右翻转还是猫""稍微变亮还是猫"。增强把这些先验"教"给模型，让它对这些变化鲁棒。
   **Injects invariances:** we *know* "a flipped cat is still a cat," "slightly brighter is still a cat." Augmentation teaches these priors so the model is robust.
3. **是一种正则化**：每个 epoch 看到的图都略有不同，模型更难过拟合到训练集的具体像素（呼应 9.10 正则化）。
   **A form of regularization:** images differ slightly each epoch, so the model can't overfit exact pixels (echoing 9.10).

> ⚠️ **铁律(面试常考)**：**只对训练集增强，不对验证/测试集**！测试要反映真实分布；测试时增强会让评估失真（推理阶段顶多用 TTA 测试时增强，那是另一回事）。另外增强必须**保标签**——比如数字 6 旋转 180° 变成 9，就不能用大角度旋转。
> ⚠️ **Iron rule (interview):** **augment only the training set, never val/test!** Test must reflect the real distribution; augmenting it distorts evaluation (inference-time TTA is separate). Also augmentation must **preserve labels** — e.g. rotating a "6" by 180° makes a "9," so avoid large rotations there.

先取一张高清彩色样例图作为"运行示例"。
Let's grab an HD color sample image as our running example.


In [ ]:
import os, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
from skimage import data, transform as sktf
from PIL import Image
sns.set_theme(style="white")

base_np = data.chelsea()                                 # skimage 自带的猫图 (300×451×3) / sample cat image
base = Image.fromarray(base_np)                          # 转成 PIL 图便于用 torchvision 变换 / to PIL
print(f"运行示例图: {base_np.shape} (高×宽×通道), 一只猫")
fig, ax = plt.subplots(figsize=(4.5, 3.2)); ax.imshow(base); ax.set_title("运行示例: chelsea (猫)"); ax.axis("off")
plt.tight_layout(); plt.show()


<a id="2"></a>
## 2. 基础增强：翻转/裁剪/旋转/调色（可视化）⭐ / Basic Augmentations

最常用的几类几何与颜色增强（用 `torchvision.transforms`）：
The most common geometric and color augmentations (via `torchvision.transforms`):
- **水平翻转(HorizontalFlip)**：左右镜像。对自然物体几乎总安全；但对文字/数字要小心。
  **Horizontal flip:** mirror. Almost always safe for natural objects; careful with text/digits.
- **随机裁剪缩放(RandomResizedCrop)**：随机裁一块再缩放回原尺寸，模拟物体不同位置/远近。
  **RandomResizedCrop:** crop a random region and resize back, simulating position/scale changes.
- **旋转(Rotation)**：小角度旋转，模拟拍摄角度变化。
  **Rotation:** small-angle rotation, simulating viewpoint changes.
- **颜色抖动(ColorJitter)**：随机改亮度/对比度/饱和度/色相，模拟光照变化。
  **ColorJitter:** randomly change brightness/contrast/saturation/hue, simulating lighting.

下面把每种增强作用在同一张图上，直观对比。
Below we apply each to the same image for visual comparison.


In [ ]:
augs = {
    "原图": lambda im: im,
    "水平翻转": transforms.RandomHorizontalFlip(p=1.0),       # p=1 一定翻转(便于展示) / always flip for demo
    "随机裁剪缩放": transforms.RandomResizedCrop((300,451), scale=(0.4,0.7)),  # 裁一块再缩放回 / crop+resize
    "旋转±25°": transforms.RandomRotation(25),
    "颜色抖动": transforms.ColorJitter(0.5, 0.5, 0.5, 0.1),   # 亮度/对比/饱和/色相 / b,c,s,h
    "灰度化": transforms.RandomGrayscale(p=1.0),
}
fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
torch.manual_seed(3)
for ax, (name, aug) in zip(axes, augs.items()):
    ax.imshow(aug(base)); ax.set_title(name, fontsize=10); ax.axis("off")
fig.suptitle("同一张图的各种增强: 内容(类别)不变, 像素变了 → 都是合法的'新'样本"); plt.tight_layout(); plt.show()
print("每种增强都保持类别不变(仍是猫), 但改变像素 → 教模型对'翻转/位移/旋转/光照'鲁棒")


In [ ]:
# "免费数据": 同一张图反复随机增强, 每次都不同 → 一张图变出很多训练样本
train_aug = transforms.Compose([                          # 实战常把多种增强组合 / compose multiple augs
    transforms.RandomHorizontalFlip(),
    transforms.RandomResizedCrop((300,451), scale=(0.5,0.9)),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.05),
])
fig, axes = plt.subplots(2, 8, figsize=(15, 4))
axes[0,0].imshow(base); axes[0,0].set_title("原图", fontsize=9); axes[0,0].axis("off")
torch.manual_seed(0)
for i, ax in enumerate(axes.ravel()):
    if i == 0: continue
    ax.imshow(train_aug(base)); ax.axis("off")           # 同一张图的15个随机增强版本 / 15 random variants
fig.suptitle("同一张图的 15 个随机增强版本: 一张图 → 大量'新'样本(免费扩充数据)"); plt.tight_layout(); plt.show()
print("每个 epoch 模型看到的同一张图都略不同 → 更难死记硬背 → 正则化效果")


<a id="3"></a>
## 3. 进阶：Mixup 与 Cutout（从零）⭐ / Advanced: Mixup & Cutout

两种现代增强，效果好且常考：
Two modern augmentations, effective and often asked:
- **Cutout**：随机**挖掉**图像中的一块（填 0/灰）。逼迫模型**不要只盯着一个局部特征**（比如不能只靠"猫脸"判断猫，被遮住时要用身体），增强鲁棒性。
  **Cutout:** randomly **mask out** a square (fill 0). Forces the model **not to rely on one local feature** (can't judge "cat" only by its face), improving robustness.
- **Mixup**：把**两张图按比例 $\lambda$ 线性混合**，标签也按同样比例混合。比如 0.6×猫 + 0.4×咖啡，标签就是 0.6 猫 + 0.4 咖啡。让决策边界更平滑、减少过度自信。
  **Mixup:** **linearly blend two images** by $\lambda$, and blend labels the same way. E.g. 0.6×cat + 0.4×coffee, label = 0.6 cat + 0.4 coffee. Smooths decision boundaries, reduces over-confidence.

下面**从零实现**这两者并可视化。
We implement both from scratch and visualize.


In [ ]:
to_tensor = transforms.ToTensor()
# 两张同尺寸图用于 Mixup / two same-size images for Mixup
catA = to_tensor(Image.fromarray(sktf.resize(data.chelsea(), (256,256), preserve_range=True).astype(np.uint8)))
cofB = to_tensor(Image.fromarray(sktf.resize(data.coffee(),  (256,256), preserve_range=True).astype(np.uint8)))

def cutout(img_t, size=80):
    """随机挖掉一个 size×size 的方块 / mask a random square."""
    c, h, w = img_t.shape; out = img_t.clone()
    y, x = np.random.randint(h), np.random.randint(w)     # 方块中心 / center
    y1, y2 = max(0,y-size//2), min(h,y+size//2)
    x1, x2 = max(0,x-size//2), min(w,x+size//2)
    out[:, y1:y2, x1:x2] = 0                              # 该区域置0 / zero the region
    return out

def mixup(img1, img2, lam):
    return lam * img1 + (1 - lam) * img2                  # 像素级加权平均 / weighted pixel average

fig, axes = plt.subplots(1, 6, figsize=(15, 2.8))
np.random.seed(1)
axes[0].imshow(catA.permute(1,2,0)); axes[0].set_title("图A: 猫", fontsize=9); axes[0].axis("off")
axes[1].imshow(cutout(catA).permute(1,2,0)); axes[1].set_title("Cutout(挖块)", fontsize=9); axes[1].axis("off")
axes[2].imshow(cutout(catA,120).permute(1,2,0)); axes[2].set_title("Cutout(更大块)", fontsize=9); axes[2].axis("off")
axes[3].imshow(cofB.permute(1,2,0)); axes[3].set_title("图B: 咖啡", fontsize=9); axes[3].axis("off")
for ax, lam in zip(axes[4:], [0.5, 0.7]):
    ax.imshow(mixup(catA, cofB, lam).permute(1,2,0))
    ax.set_title(f"Mixup λ={lam}\n{lam}猫+{1-lam:.1f}咖啡", fontsize=8); ax.axis("off")
fig.suptitle("Cutout: 挖掉局部逼模型看全局; Mixup: 两图按λ混合(标签也按λ混合)"); plt.tight_layout(); plt.show()
print("Cutout → 不依赖单一局部特征; Mixup → 平滑决策边界/减少过度自信; 都常用于提升泛化")


<a id="4"></a>
## 4. 实测：有无增强对比 + 小结 ⭐ / Experiment: With vs Without

最有说服力的是实测。我们在 **FashionMNIST 小子集**上训练同一个 CNN 两次——一次**不增强**、一次**加几何增强**——对比**训练/测试精度**与**过拟合程度(训练-测试差距)**。
The most convincing is to measure. We train the same CNN twice on a **small FashionMNIST subset** — once **without**, once **with** geometric augmentation — comparing **train/test accuracy** and the **overfitting gap**.

FashionMNIST 是灰度图，所以用**几何增强**（水平翻转 + 随机裁剪），不用颜色抖动。
FashionMNIST is grayscale, so we use **geometric augmentation** (horizontal flip + random crop), not color jitter.

> ⚠️ **诚实预期（重要的实战认知）**：增强**最可靠的效果是大幅缩小过拟合差距（正则化）**；但它**不保证一定提升测试精度**。下面的结果里，无增强模型本身已能较好泛化(FashionMNIST 较简单)，所以增强主要体现在"差距变小"，测试精度**基本持平甚至略降**。增强真正显著拉高测试精度，通常出现在：**任务更难/数据更少导致严重过拟合** + **增强恰好匹配数据的真实不变性** + **训练足够久**。乱用或过强的增强反而会掉点（如某些类并非左右对称、裁掉了关键信息）。
> ⚠️ **Honest expectation (important practical insight):** augmentation's **most reliable effect is shrinking the overfit gap (regularization)**; it does **not guarantee higher test accuracy**. Below, the no-aug model already generalizes fairly well (FashionMNIST is easy), so augmentation mainly shows as a **smaller gap**, with test accuracy **about the same or slightly lower**. Augmentation clearly lifts test accuracy mainly when: **the task is harder / data scarcer (severe overfitting)** + **augmentations match the data's true invariances** + **training is long enough**. Aggressive or mismatched augmentation can even hurt (some classes aren't left-right symmetric; cropping may remove key info).


In [ ]:
DATA_ROOT = os.path.expanduser("~/.cache/dsfs_cv")
tf_plain = transforms.Compose([transforms.ToTensor()])
tf_aug   = transforms.Compose([transforms.RandomHorizontalFlip(),
                               transforms.RandomCrop(28, padding=2),
                               transforms.ToTensor()])

def loaders(train_tf):
    tr = torchvision.datasets.FashionMNIST(DATA_ROOT, train=True, download=True, transform=train_tf)
    te = torchvision.datasets.FashionMNIST(DATA_ROOT, train=False, download=True, transform=tf_plain)  # 测试永不增强! / never augment test
    return (DataLoader(Subset(tr, range(4000)), batch_size=128, shuffle=True),
            DataLoader(Subset(te, range(2000)), batch_size=256))

def small_cnn():
    return nn.Sequential(
        nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),    # 28→14
        nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),   # 14→7
        nn.Flatten(), nn.Linear(64*7*7, 128), nn.ReLU(), nn.Linear(128, 10))

def run(train_tf, epochs=20):
    tl, el = loaders(train_tf); torch.manual_seed(0); net = small_cnn()
    opt = torch.optim.Adam(net.parameters(), lr=1e-3); ce = nn.CrossEntropyLoss()
    def acc(loader):
        net.eval(); c=t=0
        with torch.no_grad():
            for xb,yb in loader: c+=(net(xb).argmax(1)==yb).sum().item(); t+=len(yb)
        return c/t
    tr_hist, te_hist = [], []
    for _ in range(epochs):
        net.train()
        for xb,yb in tl: opt.zero_grad(); ce(net(xb),yb).backward(); opt.step()
        tr_hist.append(acc(tl)); te_hist.append(acc(el))
    return tr_hist, te_hist

tr0, te0 = run(tf_plain)      # 不增强 / no aug
tr1, te1 = run(tf_aug)        # 增强 / with aug
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(tr0, "--", color="#e67", label="train"); axes[0].plot(te0, "-", color="#e67", label="test")
axes[0].set_title(f"无增强: 训练{tr0[-1]:.2f}/测试{te0[-1]:.2f} (差距大=过拟合)"); axes[0].legend(); axes[0].set_xlabel("epoch"); axes[0].set_ylabel("准确率")
axes[1].plot(tr1, "--", color="#39c", label="train"); axes[1].plot(te1, "-", color="#39c", label="test")
axes[1].set_title(f"有增强: 训练{tr1[-1]:.2f}/测试{te1[-1]:.2f} (差距小=泛化好)"); axes[1].legend(); axes[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()
print(f"无增强: 训练 {tr0[-1]:.3f}, 测试 {te0[-1]:.3f}, 过拟合差距 {tr0[-1]-te0[-1]:.3f}")
print(f"有增强: 训练 {tr1[-1]:.3f}, 测试 {te1[-1]:.3f}, 过拟合差距 {tr1[-1]-te1[-1]:.3f}")
print(f"\n过拟合差距: {tr0[-1]-te0[-1]:.3f} → {tr1[-1]-te1[-1]:.3f} (大幅缩小, 这是增强最可靠的效果=正则化)")
print("注意: 此处无增强模型本就泛化不错, 测试精度基本持平; 增强拉高测试精度需更难任务/更少数据/更久训练+匹配的增强")


```
为什么有效: ①免费扩充数据 ②注入不变性(翻转/光照等仍是同类) ③正则化(每epoch图略变)
铁律: 只增强训练集, 绝不增强验证/测试集; 增强必须保标签(数字别大角度旋转)
基础增强: 水平翻转/随机裁剪缩放/旋转/颜色抖动/灰度化
进阶: Cutout(挖块, 逼看全局) / Mixup(两图+标签按λ混合, 平滑边界) / CutMix
效果: 最可靠是大幅缩小过拟合差距(正则化); 测试增益取决于任务难度/数据量/增强是否匹配不变性
```

### 💡 面试速查 / Interview cheat-sheet
1. **为什么有效**: 扩数据 + 注入不变性 + 正则化。
   Why: more data + injected invariances + regularization.
2. **只增强训练集**: 测试/验证绝不增强(否则评估失真)。
   Augment train only: never val/test (would distort evaluation).
3. **保标签**: 变换不能改变类别(如数字别180°旋转)。
   Label-preserving: transforms mustn't change the class.
4. **Mixup/Cutout**: 混两图+标签 / 挖块逼看全局。
   Mixup/Cutout: blend two images+labels / mask to force global view.
5. **效果**: 可靠地减小过拟合差距(正则化); 测试增益视任务/数据/增强匹配度而定。
   Effect: reliably shrinks the overfit gap (regularization); test gain depends on task/data/augmentation fit.

### 下一节 / Next
**10.6 目标检测**——前面都是"整张图是什么类别"。检测要回答"**图里有哪些物体、各在哪**"(画框)。我们会讲 bbox、IoU、NMS(从零实现)，以及 R-CNN 家族、YOLO、SSD 的核心思想。
**10.6 Object Detection** — so far "what class is the whole image." Detection answers "**which objects are where**" (draw boxes). We'll cover bbox, IoU, NMS (from scratch), and R-CNN family, YOLO, SSD.
